In [6]:
import tkinter as tk
from tkinter import filedialog, ttk, colorchooser, messagebox
from PIL import Image, ImageTk, ImageOps, EpsImagePlugin
import io
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from torchvision.models.detection import keypointrcnn_resnet50_fpn
from torchvision.models.detection.keypoint_rcnn import KeypointRCNNPredictor
import threading
import json
import os

EpsImagePlugin.gs_windows_binary = r'C:\Program Files\gs\gs10.05.0\bin\gswin64c.exe'

class CubeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.annotations = self.load_annotations()

    def load_annotations(self):
        annotations = []
        for file in os.listdir(self.root_dir):
            if file.endswith(".json"):
                with open(os.path.join(self.root_dir, file)) as f:
                    data = json.load(f)
                    annotations.append({
                        "image_path": data["image_path"],
                        "keypoints": data["keypoints"]
                    })
        return annotations

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        image = Image.open(ann["image_path"]).convert("RGB")
        keypoints = torch.tensor(ann["keypoints"], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        
        targets = {}
        targets["keypoints"] = keypoints.unsqueeze(0) 
        targets["labels"] = torch.tensor([1])
        # Простейший bbox:
        targets["boxes"] = torch.tensor([[0, 0, 1, 1]], dtype=torch.float32)
        
        return image, targets

class CubeRecognizerPro:
    def __init__(self, root):
        self.root = root
        self.root.title("Cube Recognizer Pro+")
        self.root.geometry("1000x800")
        
        self.style = ttk.Style()
        self.style.configure("TButton", font=("Helvetica", 10), padding=5)
        self.style.configure("TRadiobutton", font=("Helvetica", 9), padding=3)

        self.draw_color = "black"
        self.eraser_size = 10
        self.draw_mode = tk.StringVar(value="free")
        
        self.training_mode = False
        self.manual_points = []
        
        self.original_bg_image = None
        self.current_bg_image = None
        self.temp_shape = None
        self.start_x = None
        self.start_y = None

        self.is_training = False
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.load_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.0001)
        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
        
        self.create_widgets()
        self.setup_event_handlers()
        self.model.eval()
        self.canvas_width = 1000
        self.canvas_height = 800
        
        # self.start_auto_update()

    def load_model(self):
        model = keypointrcnn_resnet50_fpn(weights=None)
        model.roi_heads.keypoint_predictor = KeypointRCNNPredictor(512, 8)
        
        if os.path.exists("finetuned_cube_model_2.pth"):
            model.load_state_dict(
                torch.load("finetuned_cube_model_2.pth", map_location=self.device)
            )

        # if os.path.exists("model_10000_epoch_13.pth"):
        #     model.load_state_dict(
        #         torch.load("model_10000_epoch_13.pth", map_location=self.device)
        #     )

        model.to(self.device)
        return model

    def create_widgets(self):
        toolbar = ttk.Frame(self.root)
        toolbar.pack(side=tk.TOP, fill=tk.X)
        
        # Радиокнопки режимов
        modes = [
            ("Freehand", "free"),
            ("Line", "line"),
            ("Rectangle", "rect"),
            ("Circle", "circle")
        ]
        for text, mode in modes:
            rb = ttk.Radiobutton(toolbar,
                                 text=text,
                                 variable=self.draw_mode,
                                 value=mode,
                                 command=self.tool_changed)
            rb.pack(side=tk.LEFT, padx=2)

        ttk.Button(toolbar, text="Stop Train", command=self.stop_training).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Clear", command=self.clear_canvas).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Image", command=self.insert_image).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Color", command=self.choose_color).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Eraser", command=self.activate_eraser).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Predict", command=self.run_model_prediction).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Train Mode", command=self.toggle_training).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Save Data", command=self.save_training_data).pack(side=tk.LEFT, padx=5)
        ttk.Button(toolbar, text="Train Model", command=self.start_training).pack(side=tk.LEFT, padx=5)

        self.canvas_frame = ttk.Frame(self.root)
        self.canvas_frame.pack(fill=tk.BOTH, expand=True)

        self.canvas = tk.Canvas(self.canvas_frame, bg="white",
                                cursor="cross", highlightthickness=0)
        self.canvas.pack(fill=tk.BOTH, expand=True)

    def setup_event_handlers(self):
        self.canvas.bind("<ButtonPress-1>", self.start_action)
        self.canvas.bind("<B1-Motion>", self.perform_action)
        self.canvas.bind("<ButtonRelease-1>", self.end_action)
        self.canvas.bind("<Configure>", self.on_resize)

    def toggle_training(self):
        self.training_mode = not self.training_mode
        if self.training_mode:
            self.canvas.config(cursor="plus")
            self.manual_points = []
            messagebox.showinfo(
                "Train Mode ON",
                "Кликайте, чтобы расставить 8 точек.\n"
                "Порядок: (0..3 передняя грань, 4..7 задняя грань)."
            )
        else:
            self.canvas.config(cursor="cross")
            self.manual_points = []
            self.canvas.delete("manual_points")

    def save_training_data(self):
        if len(self.manual_points) != 8:
            messagebox.showerror("Error", "Нужно расставить ровно 8 точек!")
            return

        w = self.canvas.winfo_width()
        h = self.canvas.winfo_height()
        normalized = [[x / w, y / h] for (x, y) in self.manual_points]

        file_path = filedialog.asksaveasfilename(defaultextension=".json")
        if not file_path:
            return
        
        image_path = f"{os.path.splitext(file_path)[0]}.png"
        self.save_canvas_image(image_path)

        data = {
            "image_path": image_path,
            "keypoints": [[x, y, 1.0] for x, y in normalized]
        }
        with open(file_path, "w") as f:
            json.dump(data, f)
        
        messagebox.showinfo("Success", "Data saved!")

    def save_canvas_image(self, path):
        """
        Сохраняем текущее содержимое Canvas в PNG
        (аналогично тому, как в генераторе кубов указывается точный размер).
        """
        ps = self.canvas.postscript(
            colormode='color',
            x=0, y=0,
            width=self.canvas.winfo_width(),
            height=self.canvas.winfo_height()
        )
        img = Image.open(io.BytesIO(ps.encode('utf-8')))
        bg = Image.new("RGB", img.size, (255,255,255))
        bg.paste(img)
        bg.save(path)

    def start_training(self):
        if not os.path.exists("train_data"):
            messagebox.showerror("Error", "Создайте папку train_data и поместите туда .json + картинки")
            return

        dataset = CubeDataset("train_data", transform=self.transform)
        if len(dataset) == 0:
            messagebox.showerror("Error", "В train_data нет валидных .json-файлов.")
            return

        dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=self.collate_fn)
        
        def train_epoch():
            self.model.train()
            total_loss = 0.0
            for images, targets in dataloader:
                images = list(img.to(self.device) for img in images)
                targets = [{k: v.to(self.device) for k, v in t.items()} for t in targets]
                
                self.optimizer.zero_grad()
                loss_dict = self.model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                losses.backward()
                self.optimizer.step()
                
                total_loss += losses.item()
            return total_loss / len(dataloader)
        
        self.is_training = True
        threading.Thread(target=self.run_training, args=(train_epoch,)).start()

    def collate_fn(self, batch):
        return tuple(zip(*batch))

    def run_training(self, train_func):
        try:
            for epoch in range(10):
                if not self.is_training:
                    break
                loss = train_func()
                self.root.after(0, self.update_training_status, epoch+1, loss)

            if self.is_training:
                torch.save(self.model.state_dict(), "trained_model.pth")
                self.root.after(
                    0,
                    messagebox.showinfo,
                    "Training Complete",
                    "Model saved as trained_model.pth"
                )
            self.is_training = False
        except Exception as e:
            self.root.after(0, messagebox.showerror, "Training Error", str(e))
            self.is_training = False

    def update_training_status(self, epoch, loss):
        self.canvas.delete("training_text")
        self.canvas.create_text(
            100, 20,
            text=f"Epoch: {epoch} | Loss: {loss:.4f}",
            fill="red",
            tags="training_text",
            anchor="nw",
            font=("Helvetica", 13)
        )

    def stop_training(self):
        self.is_training = False
        messagebox.showinfo("Info", "Training stopped.")

    def run_model_prediction(self):
        """
        Сканируем canvas, сохраняем в temp-изображение (512x512),
        подаём в модель.
        """
        self.canvas.delete("prediction")
        try:
            target_size = (512, 512)
            ps = self.canvas.postscript(
                colormode='color',
                x=0, y=0,
                width=self.canvas.winfo_width(),
                height=self.canvas.winfo_height()
            )
            img = Image.open(io.BytesIO(ps.encode('utf-8')))
            bg = Image.new("RGB", img.size, (255,255,255))
            bg.paste(img)

            resized_img = bg.resize(target_size, Image.Resampling.LANCZOS)
            tensor = transforms.ToTensor()(resized_img).unsqueeze(0).to(self.device)
            
            threading.Thread(target=self.predict, args=(tensor, target_size)).start()
        except Exception as e:
            print(f"Ошибка: {e}")

    def predict(self, tensor, target_size):
        try:
            with torch.no_grad():
                outputs = self.model(tensor)
            self.root.after(0, self.handle_prediction, outputs, target_size)
        except Exception as e:
            print(f"Ошибка предсказания: {e}")

    def handle_prediction(self, outputs, target_size):
        self.canvas.delete("prediction")
        if len(outputs) == 0 or "keypoints" not in outputs[0]:
            return
        
        keypoints_tensor = outputs[0]["keypoints"]
        if keypoints_tensor.shape[0] == 0:
            return
        
        keypoints = keypoints_tensor[0].cpu().numpy()
        
        scale_x = self.canvas_width / target_size[0]
        scale_y = self.canvas_height / target_size[1]
        
        scaled_keypoints = []
        for (x, y, conf) in keypoints:
            scaled_keypoints.append((x * scale_x, y * scale_y, conf))
        
        self.draw_predictions(scaled_keypoints)

    def draw_predictions(self, keypoints):
        threshold = 0.5
        valid_points = [kp for kp in keypoints if kp[2] > threshold]
        
        if len(valid_points) == 8:
            edges = [
                (0,1), (1,2), (2,3), (3,0),
                (4,5), (5,6), (6,7), (7,4),
                (0,4), (1,5), (2,6), (3,7)
            ]
            for (x, y, _) in valid_points:
                self.canvas.create_oval(
                    x-4, y-4, x+4, y+4,
                    fill="#FF0000",
                    tags="prediction"
                )
            for (s,e) in edges:
                x1,y1,_ = valid_points[s]
                x2,y2,_ = valid_points[e]
                self.canvas.create_line(
                    x1, y1, x2, y2,
                    fill="#00FF00", width=2,
                    tags="prediction"
                )

    def start_action(self, event):
        if self.training_mode:
            if len(self.manual_points) < 8:
                x, y = event.x, event.y
                self.manual_points.append((x, y))
                self.canvas.create_oval(
                    x-4, y-4, x+4, y+4,
                    fill="blue", tags="manual_points"
                )
                if len(self.manual_points) == 8:
                    messagebox.showinfo("Info", "8 точек расставлено. 'Save Data'.")
            return
        
        self.start_x = event.x
        self.start_y = event.y
        mode = self.draw_mode.get()
        
        if mode == "erase":
            self.delete_items(event.x, event.y)
        elif mode in ["rect", "circle"]:
            self.create_temp_shape(event.x, event.y)

    def perform_action(self, event):
        if self.training_mode:
            return
        
        mode = self.draw_mode.get()
        if mode == "free":
            self.draw_freehand(event)
        elif mode == "line":
            self.draw_line(event)
        elif mode in ["rect", "circle"]:
            self.update_temp_shape(event)
        elif mode == "erase":
            self.delete_items(event.x, event.y)

    def end_action(self, event):
        if self.training_mode:
            return
        
        mode = self.draw_mode.get()
        if mode in ["rect", "circle"]:
            self.finalize_shape()
        self.start_x = None
        self.start_y = None
        self.temp_shape = None

    def tool_changed(self):
        self.canvas.config(cursor="cross")
        if self.draw_mode.get() in ["rect", "circle"]:
            self.canvas.config(cursor="tcross")

    def activate_eraser(self):
        self.draw_mode.set("erase")
        self.canvas.config(cursor="circle")

    def draw_freehand(self, event):
        if self.start_x is not None and self.start_y is not None:
            self.canvas.create_line(
                self.start_x, self.start_y, event.x, event.y,
                fill=self.draw_color, width=2, capstyle=tk.ROUND
            )
            self.start_x = event.x
            self.start_y = event.y

    def draw_line(self, event):
        if not self.temp_shape:
            self.temp_shape = self.canvas.create_line(
                self.start_x, self.start_y, event.x, event.y,
                fill=self.draw_color, width=2
            )
        else:
            self.canvas.coords(
                self.temp_shape,
                self.start_x, self.start_y,
                event.x, event.y
            )

    def create_temp_shape(self, x, y):
        mode = self.draw_mode.get()
        if mode == "rect":
            self.temp_shape = self.canvas.create_rectangle(
                x, y, x, y,
                outline=self.draw_color
            )
        elif mode == "circle":
            self.temp_shape = self.canvas.create_oval(
                x-1, y-1, x+1, y+1,
                outline=self.draw_color
            )

    def update_temp_shape(self, event):
        if not self.temp_shape:
            return
        mode = self.draw_mode.get()
        if mode == "rect":
            self.canvas.coords(
                self.temp_shape,
                self.start_x, self.start_y,
                event.x, event.y
            )
        elif mode == "circle":
            radius = ((event.x - self.start_x)**2 + (event.y - self.start_y)**2)**0.5
            self.canvas.coords(
                self.temp_shape,
                self.start_x - radius,
                self.start_y - radius,
                self.start_x + radius,
                self.start_y + radius
            )

    def finalize_shape(self):
        if self.temp_shape:
            self.canvas.itemconfig(self.temp_shape, outline=self.draw_color)
            if self.draw_mode.get() == "rect":
                self.canvas.itemconfig(self.temp_shape, fill="")
            self.temp_shape = None

    def delete_items(self, x, y):
        items = self.canvas.find_overlapping(
            x - self.eraser_size, y - self.eraser_size,
            x + self.eraser_size, y + self.eraser_size
        )
        for item in items:
            if "prediction" not in self.canvas.gettags(item):
                self.canvas.delete(item)

    def clear_canvas(self):
        self.canvas.delete("all")
        self.original_bg_image = None
        self.current_bg_image = None
        self.canvas.config(bg="white")
        self.draw_mode.set("free")
        self.canvas.config(cursor="cross")

    def insert_image(self):
        file_path = filedialog.askopenfilename(
            filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp")]
        )
        if file_path:
            self.original_bg_image = Image.open(file_path)
            self.redraw_background()

    def redraw_background(self):
        if not self.original_bg_image:
            return
        self.current_bg_image = ImageOps.contain(
            self.original_bg_image,
            (self.canvas_width, self.canvas_height)
        )
        self.bg_photo = ImageTk.PhotoImage(self.current_bg_image)
        self.canvas.delete("bg")
        self.canvas.create_image(
            0, 0, image=self.bg_photo,
            anchor=tk.NW, tags="bg"
        )

    def on_resize(self, event):
        new_width = max(event.width, 100)
        new_height = max(event.height, 100)
        self.canvas.config(width=new_width, height=new_height)
        self.canvas_width = new_width
        self.canvas_height = new_height
        
        if self.original_bg_image:
            self.redraw_background()

    def choose_color(self):
        color = colorchooser.askcolor()[1]
        if color:
            self.draw_color = color

    def start_auto_update(self):
        self.update_prediction()

    def update_prediction(self):
        self.run_model_prediction()
        self.root.after(50000, self.update_prediction)


if __name__ == "__main__":
    root = tk.Tk()
    app = CubeRecognizerPro(root)
    root.mainloop()


C:\Users\andre\AppData\Local\Temp\ipykernel_26772\723219364.py:115: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("finetuned_cube_model_2.pth", map_location=self.